In [ ]:
import sys; sys.path.insert(0, "..")
from pathlib import Path
import json
import torch
from PIL import Image
import matplotlib.pyplot as plt

from hanoi_caption.kb_loader import load_kb
from hanoi_caption.kb_indexer import build_or_load_index
from hanoi_caption.pipeline import caption_phase1, caption_phase2
from hanoi_caption.model_registry import registry

nodes = load_kb("../data/kb.json")
kb_index = build_or_load_index(nodes)
print(f"KB ready: {len(nodes)} nodes")

In [ ]:
photos = [
    Path("../tests/fixtures/temple_of_literature_1.jpg"),
    Path("../tests/fixtures/temple_of_literature_2.jpg"),
    Path("../tests/fixtures/temple_of_literature_3.jpg"),
]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, p in zip(axes, photos):
    ax.imshow(Image.open(p)); ax.set_title(p.name); ax.axis("off")
plt.show()

In [ ]:
phase1_results = []
for p in photos:
    img = Image.open(p).convert("RGB")
    r = caption_phase1(image=img, kb_nodes=nodes, kb_index=kb_index)
    phase1_results.append(r)
    print(f"\n--- Phase 1: {p.name} ---")
    print(r.caption or r.refusal)

In [ ]:
phase2_results = []
for p in photos:
    img = Image.open(p).convert("RGB")
    r = caption_phase2(image=img, kb_nodes=nodes, kb_index=kb_index)
    phase2_results.append(r)
    print(f"\n--- Phase 2: {p.name} ---")
    print(r.caption or r.refusal)
    print(f"  regions: {r.debug.get('n_regions')}, queries: {r.debug.get('queries')}")

In [ ]:
def jaccard(a: str, b: str) -> float:
    sa, sb = set(a.lower().split()), set(b.lower().split())
    return len(sa & sb) / len(sa | sb) if sa | sb else 0.0

print("Phase 1 pairwise word-jaccard:")
print(f"  1-2: {jaccard(phase1_results[0].caption, phase1_results[1].caption):.2f}")
print(f"  1-3: {jaccard(phase1_results[0].caption, phase1_results[2].caption):.2f}")
print(f"  2-3: {jaccard(phase1_results[1].caption, phase1_results[2].caption):.2f}")
print("\nPhase 2 pairwise word-jaccard:")
print(f"  1-2: {jaccard(phase2_results[0].caption, phase2_results[1].caption):.2f}")
print(f"  1-3: {jaccard(phase2_results[0].caption, phase2_results[2].caption):.2f}")
print(f"  2-3: {jaccard(phase2_results[1].caption, phase2_results[2].caption):.2f}")

In [ ]:
import base64, io, numpy as np
img = Image.open(photos[0]).convert("RGB")
result = caption_phase2(image=img, kb_nodes=nodes, kb_index=kb_index)
regions = result.debug["regions"]
print(f"queries fired: {result.debug['queries']}")
print(f"regions kept : {len(regions)}")
for r in regions[:6]:
    print(f"  {r['query']!r} score={r['score']:.2f}")

In [ ]:
print("loaded:", registry.loaded())
print(f"VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"VRAM peak: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")